# AI Model Validation: From Mathematics to Trust
## Episode 15 — Explainability and Interpretability: Can We Understand Why the Model Made This Decision?

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

> **A plausible explanation is not necessarily a faithful explanation.**

Scientific chain:

### Model → Prediction → Explanation Method → Attribution → Perturbation Test → Stability → Faithfulness → Human Interpretation → Evidence → Trust Decision


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import itertools, math, platform, sys
rng=np.random.default_rng(1515)
print("Python:",sys.version.split()[0],"| NumPy:",np.__version__,"|",platform.system(),platform.machine())


## 1. Interpretability and Explainability

Interpretability concerns how directly humans can understand a model mechanism. Explainability often uses methods to describe behaviour of complex models.

### Interpretable Model ≠ Post-Hoc Explanation


## 2. Prediction Is Not Explanation

A model produces:

### ŷ=f(x)

An explanation method produces:

### e=E(f,x)

Therefore the explanation is another computational object that must itself be validated.

### Explanation ≠ Model


## 3. Global, Local, Counterfactual

**Global:** How does the model behave overall?

**Local:** Why did it behave this way for this case?

**Counterfactual:** What feasible change would alter the decision?

These questions are not interchangeable.


## 4. Linear Interpretability

For:

### f(x)=β₀+Σ_jβ_jx_j

the local derivative is:

### ∂f/∂x_j=β_j

Yet coefficient magnitude depends on feature scale and does not establish causal importance.


## 5. Association Is Not Causation

If the model depends strongly on X_j, it does not follow that intervening on X_j causes Y to change.

### Feature Attribution ≠ Causal Effect


## 6. Global Feature Importance

Global importance can use coefficients, permutation, sensitivity, split importance, or aggregated local attributions.

### Feature Importance Is Method-Dependent


## 7. Permutation Importance

With baseline loss L_base and loss after permuting feature j:

### I_j=L_perm,j−L_base

Large degradation means the model relied on information destroyed by that perturbation.


In [ ]:
N=5000
X=rng.normal(0,1,(N,3))
y=3*X[:,0]-1.5*X[:,1]+.2*X[:,2]+rng.normal(0,.5,N)
def model(Z): return 3*Z[:,0]-1.5*Z[:,1]+.2*Z[:,2]
def mse(a,b): return np.mean((a-b)**2)
base=mse(y,model(X))
imp=[]
for j in range(3):
    Z=X.copy(); Z[:,j]=rng.permutation(Z[:,j])
    imp.append(mse(y,model(Z))-base)
print("Baseline MSE:",base,"Importance:",imp)
plt.figure(figsize=(7,4)); plt.bar(["X1","X2","X3"],imp)
plt.ylabel("Increase in MSE"); plt.title("Permutation Importance"); plt.grid(axis="y",alpha=.2); plt.show()


## 8. Correlated Features

Independent permutation can create implausible combinations when features are correlated and can misallocate importance among substitutes.

### Perturbation Validity Matters


## 9. Partial Dependence

For feature j:

### PD_j(z)=E_X[f(z,X_−j)]

It averages predictions after fixing feature j to z. It describes model behaviour—not causal effect.


## 10. Individual Conditional Expectation

For observation i:

### ICE_i,j(z)=f(z,x_i,−j)

and:

### PD_j(z)=1/n Σ_i ICE_i,j(z)

ICE can expose heterogeneous responses hidden by averages.


In [ ]:
def nonlinear(Z): return np.sin(Z[:,0])+.5*Z[:,1]**2+.4*Z[:,0]*Z[:,1]
Xn=rng.normal(0,1,(3000,2)); grid=np.linspace(-3,3,100)
pd=[]
for z in grid:
    Z=Xn.copy(); Z[:,0]=z; pd.append(nonlinear(Z).mean())
plt.figure(figsize=(8,5)); plt.plot(grid,pd)
plt.xlabel("X1 fixed value"); plt.ylabel("Average prediction"); plt.title("Partial Dependence"); plt.grid(alpha=.25); plt.show()


## 11. Local Sensitivity

For differentiable f:

### f(x+δ)≈f(x)+∇f(x)ᵀδ

Gradient magnitude measures local sensitivity.

### Sensitivity ≠ Semantic Importance


## 12. Gradient × Input

A simple attribution is:

### a_j=x_j∂f(x)/∂x_j

It depends on representation and baseline conventions.


## 13. Local Surrogate Explanations

Approximate f near x₀ using simpler g:

### minimize_g Σ_i π_x₀(z_i)[f(z_i)−g(z_i)]²+Ω(g)

The surrogate explains its approximation to f.

### Simplicity ≠ Faithfulness


## 14. Surrogate Fidelity

Explanation quality should include approximation error between f and g in the claimed neighborhood. A simple but inaccurate surrogate is not a trustworthy explanation.


## 15. Neighborhood Dependence

Local explanations depend on perturbation distribution, distance, kernel width, representation, and neighborhood size.

### Local Explanation = Model + Method + Neighborhood


## 16. Shapley-Value Foundation

For feature set N and feature j:

### φ_j=Σ_(S⊆N\{j}) [|S|!(|N|−|S|−1)!/|N|!]·[v(S∪{j})−v(S)]

Shapley values average marginal contribution across feature coalitions.


## 17. The Value Function Matters

The explanation must define what it means for features to be absent through a value function v(S), background distribution, conditional or marginal expectation, and masking semantics.

### Shapley Formula Alone Does Not Define the Explanation


In [ ]:
beta=np.array([2.,-1.2,.5]); x=np.array([1.5,2.,-1.]); d=3
def value(S): return sum(beta[j]*x[j] for j in S)
def shap(j):
    others=[i for i in range(d) if i!=j]; total=0.
    for r in range(len(others)+1):
        for S in itertools.combinations(others,r):
            w=math.factorial(r)*math.factorial(d-r-1)/math.factorial(d)
            total+=w*(value(set(S)|{j})-value(set(S)))
    return total
phi=np.array([shap(j) for j in range(d)])
print("Shapley:",phi,"| beta*x:",beta*x,"| sum:",phi.sum())


## 18. Shapley Efficiency

A defining property is:

### Σ_jφ_j=v(N)−v(∅)

This accounting identity does not establish causal truth.


## 19. Reference Population

Under a common expectation-based linear interpretation:

### φ_j=β_j(x_j−E[X_j])

Changing the reference population changes attribution.

### Explanation Depends on Reference Population


## 20. Correlation and Attribution

With correlated features, marginal and conditional explanations can allocate credit differently.

### Attribution Is Conditional on Explanation Assumptions


## 21. Counterfactual Explanation

Find a nearby feasible point producing a desired output:

### x*=argmin_z d(x,z) subject to f(z)=y*

Counterfactual explanation asks what would need to differ in the model input for the decision to change.


## 22. Linear Counterfactual

For linear score f(x)=wᵀx+b, the minimum L2 displacement to the zero decision boundary is:

### δ*=−f(x)w/||w||²

This is geometry, not causality.


In [ ]:
w=np.array([1.5,-.8]); b=-.4; x0=np.array([-.2,.5])
s=x0@w+b; delta=-s*w/(w@w); xcf=x0+delta
print("Original score:",s)
print("Boundary point:",xcf)
print("Boundary score:",xcf@w+b)
print("Distance:",np.linalg.norm(delta))


## 23. Counterfactual Feasibility

Counterfactuals should respect immutable attributes, physical bounds, discrete values, causal dependencies, actionability, cost, and time.

### Minimal Mathematical Change ≠ Feasible Human Action


## 24. Counterfactuals Are Not Causal Advice

Changing a model input and flipping a decision does not prove that changing that variable in reality causes the desired outcome.

### Model Counterfactual ≠ Causal Intervention


## 25. Explanation Stability

For nearby x and x':

### Stability=d_E(E(f,x),E(f,x'))/d_X(x,x')

Large explanation changes under tiny meaningful input changes can make explanations unreliable.


## 26. Repeatability

Sampling- or optimization-based explanation methods can vary across repeated runs. Explanation randomness should be measured and reported.


## 27. Faithfulness

A faithful explanation accurately reflects the model behaviour it claims to describe. Claimed important features can be perturbed and the actual prediction response measured.


## 28. Deletion Test

Let S_k be the top-k attributed features. Replace them with a stated baseline and measure:

### D(k)=|f(x)−f(x with S_k replaced)|

Large meaningful changes can support the ranking, subject to perturbation validity.


In [ ]:
beta=np.array([2.,-1.2,.5,.1]); x=np.array([1.5,2.,-1.,3.])
order=np.argsort(-np.abs(beta*x)); original=.7+x@beta; drops=[]
for k in range(1,5):
    z=x.copy(); z[order[:k]]=0; drops.append(abs(original-(.7+z@beta)))
plt.figure(figsize=(7,4)); plt.plot(range(1,5),drops,marker="o")
plt.xlabel("Top-k features removed"); plt.ylabel("|Prediction change|")
plt.title("Deletion-Style Faithfulness Test"); plt.grid(alpha=.25); plt.show()


## 29. Insertion Test

Start from a baseline and restore features in explanation-ranked order. Rapid recovery can support the ranking. Deletion and insertion are diagnostics, not universal proofs.


## 30. Sanity Checks

If model parameters are randomized, model-specific explanations should materially change. If an explanation survives destruction of learned parameters, it may reflect input structure rather than learned logic.

### Attractive Heatmap ≠ Model-Specific Explanation


## 31. No Universal Ground-Truth Explanation

For complex models, explanation validation often evaluates fidelity, stability, sensitivity, completeness, consistency, plausibility, and human usefulness. These properties are distinct.


## 32. Plausibility Versus Faithfulness

Humans may prefer familiar narratives.

### Human-Plausible Explanation ≠ Faithful Model Explanation

Persuasiveness is not scientific validation.


## 33. Completeness

Some additive methods satisfy:

### f(x)−f(x_ref)=Σ_j a_j

This is an accounting constraint, not a causal guarantee.


## 34. Sparsity

Humans often prefer few reasons:

### ||a||₀ = number of nonzero attributions

But sparsity can hide interactions.

### Simpler Explanation ≠ More Complete Explanation


## 35. Feature Interactions

For:

### f(x₁,x₂)=x₁x₂

neither feature has a fixed standalone contribution independent of the other. Attribution requires an allocation convention.


In [ ]:
grid=np.linspace(-2,2,100)
X1,X2=np.meshgrid(grid,grid); Z=X1*X2
plt.figure(figsize=(7,5)); plt.contourf(X1,X2,Z,levels=30)
plt.xlabel("X1"); plt.ylabel("X2"); plt.title("Interaction Surface: X1 × X2"); plt.colorbar(label="Prediction"); plt.show()


## 36. Global vs Local

A feature with low average importance can dominate a narrow high-risk region.

### Global Importance ≠ Local Importance

Likewise, a local explanation at x₀ should not be generalized to a population without evidence.


## 37. Explanation Drift

Under changing deployment populations, the distribution of explanations can change:

### E_t₀ vs E_t₁

Explanation drift is an investigation signal, not proof of causal change.


## 38. Subgroup Explanations

Compare explanation behaviour conditionally:

### E[E(f,X)|G=A] vs E[E(f,X)|G=B]

Interpretation must account for different feature distributions and sample sizes.


## 39. Explainability and Fairness

Background choice, sparse subgroup data, correlation, representation, and neighborhood construction can make explanation methods behave differently across groups.

### Fair Model Evaluation Requires Explanation-Method Evaluation


## 40. Explainability and Safety

Explanations can support failure investigation, human review, incident analysis, control verification, and auditability. They do not replace direct safety, reliability, robustness, or performance evidence.


## 41. Transparency, Privacy, Security

Detailed explanations may reveal sensitive data, model behaviour, intellectual property, or attack-relevant information. Explanation exposure is a system-design decision.


## 42. Human Evaluation

Measure comprehension, decision quality, time to understand, error detection, over-trust, and appropriate reliance—not merely whether users like the explanation.


## 43. Explanations Can Increase Over-Trust

A polished explanation can make a wrong prediction feel justified. Compare human decisions with and without explanations and test whether appropriate reliance improves.


## 44. Explanation Validation Matrix

Record scope, target, baseline, perturbation semantics, fidelity, stability, repeatability, completeness, sparsity, correlation sensitivity, causal limitations, human usefulness, and privacy/security implications.


## 45. OpenVals Perspective

A validation platform should preserve explanation method/version, model/version, target prediction, reference data, perturbation distribution, attributions, surrogate fidelity, stability, repeatability, deletion/insertion evidence, sanity checks, subgroup behaviour, explanation drift, counterfactual feasibility, causal limitations, and human-evaluation evidence.

The stronger question is:

### What evidence shows that these explanations faithfully represent the model behaviour they claim to explain?


## 46. Scientific Explainability Chain

### Model → Prediction → Explanation Method → Attribution → Perturbation Test → Stability → Faithfulness → Human Interpretation → Evidence → Trust Decision

Explainability becomes an object of validation rather than a decorative AI layer.


## 47. Mathematical Summary

### ŷ=f(x)

### e=E(f,x)

### I_j=L_perm,j−L_base

### PD_j(z)=E_X[f(z,X_−j)]

### ICE_i,j(z)=f(z,x_i,−j)

### f(x+δ)≈f(x)+∇f(x)ᵀδ

### a_j=x_j∂f/∂x_j

### φ_j=Σ_(S⊆N\{j})w(S)[v(S∪{j})−v(S)]

### Σ_jφ_j=v(N)−v(∅)

### x*=argmin_z d(x,z) subject to f(z)=y*

### Stability=d_E(E(f,x),E(f,x'))/d_X(x,x')

### Completeness: f(x)−f(x_ref)=Σ_j a_j


## 48. Central Conclusion

### Prediction ≠ Explanation
### Attribution ≠ Causation
### Plausibility ≠ Faithfulness
### Simplicity ≠ Truth
### Explanation Availability ≠ Explanation Validity

> **An explanation should be trusted only to the extent that evidence shows it faithfully represents the model behaviour it claims to explain.**

This gives OpenVals another core principle:

> **Explainability itself must be validated.**

## Next Episode

**Episode 16 — Consistency, Reproducibility, and Variance: Will the AI Give Us the Same Quality Again?**

Episode 16 can develop repeated-run variance, stochastic inference, seeds, temperature, inter-run agreement, variance decomposition, confidence intervals, reproducibility, prompt sensitivity, model-version variability, and why a single successful run is weak evidence.
